# Etsy API walkthrough

A hands-on tour of how **Etsy Listing Scout** fetches data from the Etsy Open API, from one raw API call up to the finished spreadsheet.

**Who it's for:** developers new to this codebase. Run it top to bottom, then change the `KEYWORD` and explore.

**What you'll learn**
1. How the code is organised (`EtsyApi` and its three layers)
2. What Etsy returns for a search, a listing, a shop and reviews, *before* any filtering
3. How prices work: seller currencies, USD conversion and sale prices
4. Popularity and material fields (`views`, `num_favorers`, `tags`, `materials`)
5. How the pipeline turns all of this into the `competitor_listing` sheet

> Every cell that calls Etsy says how many API calls it makes. Etsy rate-limits keys, so keep `limit` small while exploring.

## 0. Before you start

| Need | How |
|---|---|
| Dependencies | `uv sync` in the project folder |
| An **active** Etsy API key | in `../.env` as `ETSY_API_KEY` and `ETSY_SHARED_SECRET` (see `docs/getting-started.md`) |
| The right kernel | this project's `.venv`, shown as *etsy-listing-scout (3.12)* |

The package is installed in *editable* mode, so import it as `etsy_scout`. Never use `src.etsy_scout`, and never use relative imports like `.export`. `autoreload` picks up your edits in `src/` without a kernel restart.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

from etsy_scout.etsy_api import EtsyApi
from etsy_scout.export import to_dataframe, write_outputs
from etsy_scout.fx import money
from etsy_scout.pipeline import run

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

KEYWORD = "adhd notebook"  # change me

## 1. How the code is organised

All API code lives in `src/etsy_scout/etsy_api.py`, in one class: `EtsyApi`. It has **three layers**:

| Layer | Methods | Returns | Use it when... |
|---|---|---|---|
| 1. Low-level | `_get(path, params)` | raw JSON for **any** endpoint | you want an endpoint the class doesn't wrap yet |
| 2. Raw endpoints | `search_page()`, `listings_batch()`, `shop()`, `listing_review_stats()` | Etsy's JSON as Python dicts, unfiltered | **exploring the data (most of this notebook)** |
| 3. Converters | `search_hits()`, `listing_details()`, `shop_stats()` | clean models (`SearchHit`, `ListingDetails`, `ShopStats`) | building features |

`_get` handles the API key header, throttling (at most about 4 calls per second) and retries on HTTP 429. It also raises `EtsyApiError` with a readable message on 401/403.

Then `pipeline.run()` chains the converters into spreadsheet rows:

```
search_hits ──► listing_details ──► shop_stats ──► build_row ──► to_dataframe / write_outputs
 (rank)          (prices, media,      (shop age,     (merge into     (Excel + CSV)
                  reviews, tags)       sales...)      ListingRow)
```

More detail: `docs/developer/architecture.md`.

## 2. Connect and check the key
`EtsyApi.from_env()` reads `ETSY_API_KEY` and `ETSY_SHARED_SECRET` from the environment. `load_dotenv()` loads them from `../.env`. **1 API call.**

In [ ]:
load_dotenv()
api = EtsyApi.from_env()
assert api is not None, "No ETSY_API_KEY found - see docs/getting-started.md"

api._get("/openapi-ping")  # returns {'application_id': ...} when the key is active

## 3. Search: the raw response
`search_page` calls `GET /listings/active`, sorted by relevance (`sort_on=score`). The response has:
- `count`: the total number of active listings matching the keyword, which is the market size
- `results`: one dict per listing

**1 API call.**

In [ ]:
page = api.search_page(KEYWORD, limit=5)
print("total matching listings:", page["count"])
print("fields per listing:", len(page["results"][0]))

first = page["results"][0]
pd.Series(first).to_frame("value")  # every field Etsy returns from search

**Things to notice**
- `listing_type` is `physical`, `download` (digital) or `both`. The app drops `download` unless you pass `--include-digital`.
- `price` is a *money object* `{"amount", "divisor", "currency_code"}` in **the seller's currency**. See section 5.
- `views`, `num_favorers`, `tags`, `materials`, `who_made`, `when_made` and `taxonomy_id` are all here already.

In [ ]:
# The top 5 at a glance
pd.DataFrame(page["results"])[
    ["listing_id", "title", "listing_type", "price", "views", "num_favorers"]
]

## 4. One listing in depth
`listings_batch` calls `GET /listings/batch` for up to 100 IDs per call, with `includes=Images,Videos,Shop,BuyerPrice` and `buyer_country=US`. This adds four blocks:

| Key | What it is |
|---|---|
| `images`, `videos` | the photo and video gallery. The app counts them for `num_photos_video` |
| `shop` | the seller's shop (name, sales, ratings...) |
| `buyer_price` | the price a **US shopper** sees: regular price, sale price, discount and shipping |

**1 API call.**

In [ ]:
listing_id = str(first["listing_id"])
full = api.listings_batch([listing_id])[listing_id]

print("keys added by the batch call:", sorted(set(full) - set(first)))
print("images:", len(full.get("images") or []), "| videos:", len(full.get("videos") or []))
pprint(full["buyer_price"])

## 5. Prices: currency, conversion and sales

Etsy returns prices in **each seller's own currency**, so a UK shop's `11.99` is £11.99, not $11.99. The research sheet compares prices in US dollars, so the app converts them with `etsy_scout.fx.FxRates`, using daily European Central Bank rates from [frankfurter.dev](https://frankfurter.dev). That service is free and needs no key, and the rates are fetched once per run.

`buyer_price` fields used by the app:

| Field | Used for |
|---|---|
| `original_price` | `price_dollar` (converted to USD) plus `price_original` and `price_currency` |
| `has_discount`, `discounted_price` | `on_sales_price_dollar` (converted to USD) |
| `discount_percentage` | `discount_pct` |
| `shipping_cost`, `is_free_shipping` | `shipping_cost_usd`, `free_shipping` |

**1 API call** for the search below. The exchange rates are 1 extra call to frankfurter.dev, not Etsy.

In [ ]:
results = api.search_page(KEYWORD, limit=20)["results"]

prices = pd.DataFrame(
    [
        {"listing_id": r["listing_id"], "amount": m[0], "currency": m[1]}
        for r in results
        if (m := money(r["price"]))
    ]
)
prices["usd"] = prices.apply(lambda p: api.fx.to_usd(p.amount, p.currency), axis=1)
print("rates date:", api.fx.date)
print(prices.currency.value_counts().to_dict())
prices

In [ ]:
# Find listings on sale (1 API call for up to 100 listings)
ids = [str(r["listing_id"]) for r in results]
batch = api.listings_batch(ids)

on_sale = [
    {
        "listing_id": lid,
        "regular": money(b["buyer_price"]["original_price"]),
        "sale": money(b["buyer_price"]["discounted_price"]),
        "discount_pct": b["buyer_price"]["discount_percentage"],
    }
    for lid, b in batch.items()
    if (b.get("buyer_price") or {}).get("has_discount")
]
print(f"{len(on_sale)} of {len(batch)} listings are on sale")
pd.DataFrame(on_sale)

## 6. Popularity and materials
These fields help judge demand and product fit:

| Field | Meaning | Tip |
|---|---|---|
| `views` | lifetime listing views | compare listings of similar age (`original_creation_timestamp`) |
| `num_favorers` | shoppers who saved the listing | a strong interest signal, often better than views |
| `tags` | up to 13 seller keywords | mine them for keyword ideas |
| `materials` | seller-entered materials | often empty; when filled, great for physical products such as jewelry |

No new API calls; this reuses `batch` from section 5.

In [ ]:
popularity = pd.DataFrame(
    [
        {
            "listing_id": lid,
            "title": b["title"][:50],
            "views": b.get("views"),
            "num_favorers": b.get("num_favorers"),
            "materials": ", ".join(b.get("materials") or []),
            "n_tags": len(b.get("tags") or []),
        }
        for lid, b in batch.items()
    ]
).sort_values("num_favorers", ascending=False)
popularity

In [ ]:
# Most common tags across the results: keyword ideas for your own listings
pd.Series([t.lower() for b in batch.values() for t in b.get("tags") or []]).value_counts().head(15)

## 7. Shop and reviews
- `api.shop(shop_id)` → `GET /shops/{id}`: sales, review average, active listings, opening date. **1 call.**
- `api.listing_review_stats(listing_id)` → `GET /listings/{id}/reviews`: (review count, average rating). **1 call.**

In [ ]:
shop = api.shop(first["shop_id"])
pd.Series(
    {
        k: shop.get(k)
        for k in [
            "shop_name",
            "transaction_sold_count",
            "review_count",
            "review_average",
            "listing_active_count",
            "create_date",
            "shop_location_country_iso",
            "is_vacation",
        ]
    }
).to_frame("value")

In [ ]:
api.listing_review_stats(listing_id)  # (count, average rating)

## 8. Converters: the cleaned versions
Layer 3 turns the raw JSON from sections 3–7 into typed models with exactly the fields the app needs. This is the code behind the spreadsheet columns.

About **8 API calls**: 1 search, 1 batch, 5 reviews and 1 shop.

In [ ]:
hits = api.search_hits(KEYWORD, top_n=5, exclude_digital=True)
hits[0]  # SearchHit: rank, listing id, url, title, shop id

In [ ]:
details = api.listing_details([h.listing_id for h in hits])
details[hits[0].listing_id]  # ListingDetails: USD prices, media, reviews, tags, materials...

In [ ]:
api.shop_stats(hits[0].shop_id)  # ShopStats: age in years, totals, rating

## 9. The full pipeline
`run()` does everything above for the top N listings, and `to_dataframe` orders the columns like the research sheet. The first 15 columns are the template; the extras come after.

About **2 + 2N API calls**. With `top_n=5`, that's roughly 12.

In [ ]:
rows = run(KEYWORD, top_n=5, source="api")
df = to_dataframe(rows)
df

In [ ]:
# Save like the CLI does. Paths are relative to notebooks/, so go up one level.
xlsx, csv = write_outputs(rows, KEYWORD, Path("..") / "output")
print(xlsx)

## 10. Try it yourself
1. Change `KEYWORD` (try `"14k gold filled necklace"`) and rerun sections 3–6. How do currencies, sales and materials differ?
2. Compare two keywords: run `run()` for each and compare the median `price_dollar` and `num_favorers`.
3. Which `materials` appear on the most-favorited listings?
4. Use `api._get(...)` to call an endpoint the class doesn't wrap yet, for example `/shops/{shop_id}/listings/active`. Check the [Etsy API reference](https://developers.etsy.com/documentation/reference) for API-key endpoints.

**Next:** `docs/user-guide/top-listings.md` (all output columns), `docs/developer/architecture.md`, and `docs/developer/contributing.md` (how to add a feature).